In [1]:

!git clone https://github.com/udayasherkar94-prog/Applied-Machine-Learning.git

Cloning into 'Applied-Machine-Learning'...
remote: Enumerating objects: 187, done.
remote: Counting objects: 100% (187/187), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 187 (delta 41), reused 118 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (187/187), 5.58 MiB | 2.15 MiB/s, done.
Resolving deltas: 100% (41/41), done.


In [2]:
%cd /content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model

%cd notebooks

/content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model
/content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model/notebooks


# 03 – Feature Engineering and Feature Selection

**Project:** Ames Housing Price Prediction  
**Goal:** Prepare the Ames Housing data for a regression model, preserve a cleaned copy with all original fields, then create a model-ready dataset containing only selected numeric features.

## Notebook workflow

1. Load the original raw dataset.
2. Clean missing values while preserving all 82 columns.
3. Save the cleaned, readable dataset to `data/cleaned/ames_cleaned.csv`.
4. Split data into training and testing sets.
5. Engineer useful features.
6. Impute and encode features using training data only.
7. Standardize encoded features and select useful features using LassoCV fitted on training data only.
8. Save selected numeric training, testing, and combined datasets in `data/processed/`.

> **Important:** The cleaned CSV is a readable, all-fields snapshot for inspection. To avoid data leakage, the model preprocessing and feature selection

## 1. Import libraries

We use pandas and NumPy for data handling, Matplotlib and Seaborn for visual checks, and scikit-learn for splitting, preprocessing, encoding, and feature selection.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LassoCV

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42

## 2. Set project paths

This notebook assumes that the GitHub repository has been cloned in the current Google Colab runtime and that the original dataset is stored in `data/raw/AmesHousing.csv`.

If your folder is in a different location, update `PROJECT_DIR` below.

In [4]:
PROJECT_DIR = Path(
    "/content/Applied-Machine-Learning/"
    "SupervisedLearning/LinearRegression/UsingScikit-Learn/"
    "03_Regression/House_Price_Prediction_Model"
)

RAW_DATA_PATH = PROJECT_DIR / "data" / "raw" / "AmesHousing.csv"
CLEANED_DIR = PROJECT_DIR / "data" / "cleaned"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Create output folders if they do not already exist.
CLEANED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw dataset exists:", RAW_DATA_PATH.exists())
print("Project directory:", PROJECT_DIR)

Raw dataset exists: True
Project directory: /content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model


## 3. Load the raw dataset

Keep the original dataset unchanged. All operations in this notebook are performed on copies.

In [5]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {RAW_DATA_PATH}\n"
        "Check that the repository is cloned and AmesHousing.csv is in data/raw/."
    )

raw_df = pd.read_csv(RAW_DATA_PATH)
df = raw_df.copy()

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2930, 82)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.0,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.0,Unf,0.0,441.0,1080.0,GasA,Fa,Y,SBrkr,1656,0,0,1656,1.0,0.0,1,0,3,1,TA,7,Typ,2,Gd,Attchd,1960.0,Fin,2.0,528.0,TA,TA,P,210,62,0,0,0,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,NaN,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,NaN,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,NaN,0.0,Gd,TA,CBlock,TA,TA,No,ALQ,1065.0,Unf,0.0,1045.0,2110.0,GasA,Ex,Y,SBrkr,2110,0,0,2110,1.0,0.0,2,1,3,1,Ex,8,Typ,2,TA,Attchd,1968.0,Fin,2.0,522.0,TA,TA,Y,0,0,0,0,0,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,NaN,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [6]:
# Check column names, data types, and non-null counts.
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   object 
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot Shape        2930 non-null   object 
 9   Land Contour     2930 non-null   object 
 10  Utilities        2930 non-null   object 
 11  Lot Config       2930 non-null   object 
 12  Land Slope       2930 non-null   object 
 13  Neighborhood     2930 non-null   object 
 14  Condition 1      2930 non-null   object 
 15  Condition 2      2930 non-null   object 
 16  Bldg Type        2930 non-null   object 
 17  House Style   

In [7]:
# Confirm that the target column is available.
TARGET = "SalePrice"

if TARGET not in df.columns:
    raise KeyError(f"Target column '{TARGET}' was not found.")

print("Target summary:")
display(df[TARGET].describe())

Target summary:


,SalePrice
count,2930.000000
mean,180796.060068
std,79886.692357
min,12789.000000
25%,129500.000000
50%,160000.000000
75%,213500.000000
max,755000.000000


## 4. Initial data quality checks

We check duplicate rows, missing values, and the target column before making changes.

In [8]:
# Count completely duplicated rows.
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

# Review missing values.
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary.head(20))

Duplicate rows: 0


,missing_count,missing_percent
Pool QC,2917,99.56
Misc Feature,2824,96.38
Alley,2732,93.24
Fence,2358,80.48
Mas Vnr Type,1775,60.58
Fireplace Qu,1422,48.53
Lot Frontage,490,16.72
Garage Qual,159,5.43
Garage Yr Blt,159,5.43
Garage Cond,159,5.43


<details>
<summary><strong>Observation notes – data quality</strong></summary>

- Review the duplicate count before deciding whether to remove rows.
- In the Ames dataset, many missing categorical values can mean that a feature is absent (for example, no pool or no alley access), rather than an unknown value.
- Check the meaning of each column before filling or dropping missing values.
- `SalePrice` is the prediction target. Rows without a target cannot be used for supervised model training.
</details>

## 5. Clean the all-fields dataset

The goal of this section is to create a readable cleaned copy while retaining all original columns.

Cleaning decisions in this notebook:
- Remove exact duplicate rows, if any.
- Remove rows where `SalePrice` is missing because the target is required for supervised learning.
- For categorical columns, use `Not Available` for missing entries. This preserves the fact that the value was not recorded; it does not assume that every missing value means the feature is absent.
- For numerical columns, fill missing values with the column median.
- Do not drop `Order` or `PID` from this cleaned copy. They are retained for traceability, but will not be used as model predictors.

The median values used in this archival cleaned copy are calculated across the available dataset. The model pipeline later fits its own imputers on training data only

In [9]:
# Work on a fresh copy so the raw DataFrame remains unchanged.
cleaned_df = raw_df.copy()

# Remove exact duplicate rows.
cleaned_df = cleaned_df.drop_duplicates().copy()

# Rows without the target cannot be used for supervised learning.
cleaned_df = cleaned_df.dropna(subset=[TARGET]).copy()

# Separate categorical and numerical columns.
categorical_cols_all = cleaned_df.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_cols_all = cleaned_df.select_dtypes(include=np.number).columns.tolist()

# Fill categorical missing values with a clear label.
for column in categorical_cols_all:
    cleaned_df[column] = cleaned_df[column].fillna("Not Available")

# Fill numerical missing values with each column's median.
for column in numerical_cols_all:
    if cleaned_df[column].isna().any():
        median_value = cleaned_df[column].median()
        cleaned_df[column] = cleaned_df[column].fillna(median_value)

print("Raw shape:", raw_df.shape)
print("Cleaned shape:", cleaned_df.shape)
print("Remaining missing values:", cleaned_df.isna().sum().sum())


Raw shape: (2930, 82)
Cleaned shape: (2930, 82)
Remaining missing values: 0


In [10]:
# Verify that all original fields are still present.
missing_columns_after_cleaning = set(raw_df.columns) - set(cleaned_df.columns)
extra_columns_after_cleaning = set(cleaned_df.columns) - set(raw_df.columns)

print("Original columns missing from cleaned data:", missing_columns_after_cleaning)
print("Unexpected extra columns:", extra_columns_after_cleaning)
print("All original columns preserved:", cleaned_df.columns.tolist() == raw_df.columns.tolist())

display(cleaned_df.head())

Original columns missing from cleaned data: set()
Unexpected extra columns: set()
All original columns preserved: True


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,Not Available,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.0,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.0,Unf,0.0,441.0,1080.0,GasA,Fa,Y,SBrkr,1656,0,0,1656,1.0,0.0,1,0,3,1,TA,7,Typ,2,Gd,Attchd,1960.0,Fin,2.0,528.0,TA,TA,P,210,62,0,0,0,0,Not Available,Not Available,Not Available,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,Not Available,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,Not Available,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,Not Available,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,Not Available,MnPrv,Not Available,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,Not Available,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,Not Available,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,Not Available,Not Available,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,Not Available,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,Not Available,0.0,Gd,TA,CBlock,TA,TA,No,ALQ,1065.0,Unf,0.0,1045.0,2110.0,GasA,Ex,Y,SBrkr,2110,0,0,2110,1.0,0.0,2,1,3,1,Ex,8,Typ,2,TA,Attchd,1968.0,Fin,2.0,522.0,TA,TA,Y,0,0,0,0,0,0,Not Available,Not Available,Not Available,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,Not Available,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,Not Available,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,Not Available,MnPrv,Not Available,0,3,2010,WD,Normal,189900


## 6. Save the cleaned dataset – all fields

This is the first requested output. It keeps the original column names and readable categorical values.

**Output:** `data/cleaned/ames_cleaned.csv`

In [11]:
CLEANED_DATA_PATH = CLEANED_DIR / "ames_cleaned.csv"

cleaned_df.to_csv(CLEANED_DATA_PATH, index=False)

print("Cleaned dataset saved to:", CLEANED_DATA_PATH)
print("Saved shape:", cleaned_df.shape)

Cleaned dataset saved to: /content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model/data/cleaned/ames_cleaned.csv
Saved shape: (2930, 82)


In [12]:
# Read the saved file back to verify it was written correctly.
cleaned_check = pd.read_csv(CLEANED_DATA_PATH)

print("Saved file shape:", cleaned_check.shape)
print("Saved file columns:", len(cleaned_check.columns))
display(cleaned_check.head())

Saved file shape: (2930, 82)
Saved file columns: 82


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,Not Available,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,5,1960,1960,Hip,CompShg,BrkFace,Plywood,Stone,112.0,TA,TA,CBlock,TA,Gd,Gd,BLQ,639.0,Unf,0.0,441.0,1080.0,GasA,Fa,Y,SBrkr,1656,0,0,1656,1.0,0.0,1,0,3,1,TA,7,Typ,2,Gd,Attchd,1960.0,Fin,2.0,528.0,TA,TA,P,210,62,0,0,0,0,Not Available,Not Available,Not Available,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,Not Available,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Feedr,Norm,1Fam,1Story,5,6,1961,1961,Gable,CompShg,VinylSd,VinylSd,Not Available,0.0,TA,TA,CBlock,TA,TA,No,Rec,468.0,LwQ,144.0,270.0,882.0,GasA,TA,Y,SBrkr,896,0,0,896,0.0,0.0,1,0,2,1,TA,5,Typ,0,Not Available,Attchd,1961.0,Unf,1.0,730.0,TA,TA,Y,140,0,0,0,120,0,Not Available,MnPrv,Not Available,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,Not Available,IR1,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,6,6,1958,1958,Hip,CompShg,Wd Sdng,Wd Sdng,BrkFace,108.0,TA,TA,CBlock,TA,TA,No,ALQ,923.0,Unf,0.0,406.0,1329.0,GasA,TA,Y,SBrkr,1329,0,0,1329,0.0,0.0,1,1,3,1,Gd,6,Typ,0,Not Available,Attchd,1958.0,Unf,1.0,312.0,TA,TA,Y,393,36,0,0,0,0,Not Available,Not Available,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,Not Available,Reg,Lvl,AllPub,Corner,Gtl,NAmes,Norm,Norm,1Fam,1Story,7,5,1968,1968,Hip,CompShg,BrkFace,BrkFace,Not Available,0.0,Gd,TA,CBlock,TA,TA,No,ALQ,1065.0,Unf,0.0,1045.0,2110.0,GasA,Ex,Y,SBrkr,2110,0,0,2110,1.0,0.0,2,1,3,1,Ex,8,Typ,2,TA,Attchd,1968.0,Fin,2.0,522.0,TA,TA,Y,0,0,0,0,0,0,Not Available,Not Available,Not Available,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,Not Available,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,2Story,5,5,1997,1998,Gable,CompShg,VinylSd,VinylSd,Not Available,0.0,TA,TA,PConc,Gd,TA,No,GLQ,791.0,Unf,0.0,137.0,928.0,GasA,Gd,Y,SBrkr,928,701,0,1629,0.0,0.0,2,1,3,1,TA,6,Typ,1,TA,Attchd,1997.0,Fin,2.0,482.0,TA,TA,Y,212,34,0,0,0,0,Not Available,MnPrv,Not Available,0,3,2010,WD,Normal,189900


## 7. Prepare features and target

`SalePrice` is the value we want to predict. `Order` and `PID` are identifiers, not meaningful property characteristics for this model, so we keep them separate and exclude them from predictors.

We split the data before fitting feature transformations. This helps prevent information from the test set leaking into the training process.

In [13]:
# Keep identifiers for reference in exported files.
id_columns = [column for column in ["Order", "PID"] if column in cleaned_df.columns]
ids = cleaned_df[id_columns].copy()

# Remove target and identifiers from predictor columns.
X = cleaned_df.drop(columns=[TARGET] + id_columns)
y = cleaned_df[TARGET].copy()

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)
print("Identifier columns:", id_columns)

Predictor shape: (2930, 79)
Target shape: (2930,)
Identifier columns: ['Order', 'PID']


In [14]:
# Split before fitting imputers, encoders, scaling, or feature selection.
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X,
    y,
    ids,
    test_size=0.20,
    random_state=RANDOM_STATE
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

Training rows: 2344
Testing rows: 586


# 8. Feature engineering

Create a small number of interpretable features from existing columns. `House Age` and `Years Since Remodel` use the sale year as the reference year.

We remove the source year columns from the predictor table after creating the derived features to avoid keeping redundant representations. The original year columns remain untouched in `ames_cleaned.csv`.

In [15]:
def add_engineered_features(data):
    data = data.copy()

    if {"Yr Sold", "Year Built"}.issubset(data.columns):
        data["House Age"] = data["Yr Sold"] - data["Year Built"]

    if {"Yr Sold", "Year Remod/Add"}.issubset(data.columns):
        data["Years Since Remodel"] = data["Yr Sold"] - data["Year Remod/Add"]

    # Remove impossible negative values caused by inconsistent dates.
    for column in ["House Age", "Years Since Remodel"]:
        if column in data.columns:
            data.loc[data[column] < 0, column] = np.nan

    # Drop source columns used to calculate the new age features.
    columns_to_drop = [
        column for column in ["Year Built", "Year Remod/Add"]
        if column in data.columns
    ]
    data = data.drop(columns=columns_to_drop)

    return data

X_train_eng = add_engineered_features(X_train)
X_test_eng = add_engineered_features(X_test)

# Keep training and testing columns aligned.
X_test_eng = X_test_eng.reindex(columns=X_train_eng.columns)

print("Training features after engineering:", X_train_eng.shape)
print("Testing features after engineering:", X_test_eng.shape)
display(X_train_eng.head())

Training features after engineering: (2344, 79)
Testing features after engineering: (586, 79)


,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,House Age,Years Since Remodel
381,20,RL,80.0,10400,Pave,Not Available,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,Norm,1Fam,1Story,7,5,Gable,CompShg,HdBoard,HdBoard,BrkFace,189.0,TA,TA,CBlock,Gd,TA,No,Unf,0.0,Unf,0.0,1090.0,1090.0,GasA,TA,Y,SBrkr,1370,0,0,1370,0.0,0.0,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2.0,479.0,TA,TA,Y,0,0,0,0,0,0,Not Available,MnPrv,Not Available,0,6,2009,WD,Family,33.0,33.0
834,60,RL,68.0,28698,Pave,Not Available,IR2,Low,AllPub,CulDSac,Sev,ClearCr,Norm,Norm,1Fam,2Story,5,5,Flat,Tar&Grv,Plywood,Plywood,Not Available,0.0,TA,TA,PConc,TA,Gd,Gd,LwQ,249.0,ALQ,764.0,0.0,1013.0,GasA,TA,Y,SBrkr,1160,966,0,2126,0.0,1.0,2,1,3,1,TA,7,Min2,0,Not Available,Attchd,1967.0,Fin,2.0,538.0,TA,TA,Y,486,0,0,0,225,0,Not Available,Not Available,Not Available,0,6,2009,WD,Abnorml,42.0,42.0
1898,90,RL,70.0,9842,Pave,Not Available,Reg,Lvl,AllPub,FR2,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,Gable,CompShg,HdBoard,HdBoard,Not Available,0.0,TA,TA,Slab,Not Available,Not Available,Not Available,Not Available,0.0,Not Available,0.0,0.0,0.0,GasA,TA,Y,SBrkr,1224,0,0,1224,0.0,0.0,2,0,2,2,TA,6,Typ,0,Not Available,CarPort,1962.0,Unf,2.0,462.0,TA,TA,Y,0,0,0,0,0,0,Not Available,Not Available,Not Available,0,3,2007,WD,Normal,45.0,45.0
678,90,RL,60.0,7200,Pave,Not Available,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,Gable,CompShg,BrkFace,Stone,Not Available,0.0,TA,TA,Slab,Not Available,Not Available,Not Available,Not Available,0.0,Not Available,0.0,0.0,0.0,Wall,Fa,N,FuseF,1040,0,0,1040,0.0,0.0,2,0,2,2,TA,6,Typ,0,Not Available,Detchd,1956.0,Unf,2.0,420.0,TA,TA,Y,0,0,0,0,0,0,Not Available,Not Available,Not Available,0,6,2009,WD,Normal,60.0,59.0
700,190,RM,63.0,7627,Pave,Not Available,Reg,Lvl,AllPub,Corner,Gtl,OldTown,Artery,Norm,2fmCon,2Story,4,6,Gable,CompShg,AsbShng,AsbShng,Not Available,0.0,Fa,TA,BrkTil,Fa,Po,No,Unf,0.0,Unf,0.0,600.0,600.0,GasA,Gd,N,SBrkr,1101,600,0,1701,0.0,0.0,2,0,4,2,Fa,8,Typ,0,Not Available,Not Available,1979.0,Not Available,0.0,0.0,Not Available,Not Available,N,0,0,148,0,0,0,Not Available,Not Available,Not Available,0,10,2009,WD,Normal,89.0,59.0


<details>
<summary><strong>Observation notes – feature engineering</strong></summary>

- `House Age` estimates the age of the property at the time it was sold.
- `Years Since Remodel` estimates how long it had been since the last recorded remodel.
- Negative values are treated as missing because they indicate inconsistent year information.
- These engineered columns are created separately for training and testing data using the same function.
</details>

## 9. Identify numerical and categorical predictors

We identify column types from the training data. The same column lists are then used to transform the test data.

In [17]:
numeric_features = X_train_eng.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train_eng.select_dtypes(exclude=np.number).columns.tolist()

print("Number of numerical features:", len(numeric_features))
print("Number of categorical features:", len(categorical_features))

Number of numerical features: 36
Number of categorical features: 43


## 10. Build preprocessing pipelines

Numerical columns:
- Fill missing values using the training median.
- Standardize numeric values.

Categorical columns:
- Fill missing values using the most frequent training value.
- Convert categories into numeric indicator columns using one-hot encoding.
- Ignore categories that appear in the test set but were not present in training.

The preprocessing object is fitted only on `X_train_eng`.

In [18]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

 If your Colab environment has an older scikit-learn version that does not recognize `sparse_output=False`, replace it with `sparse=False`.

In [19]:
# Fit only on training data, then transform both splits.
X_train_encoded = preprocessor.fit_transform(X_train_eng)
X_test_encoded = preprocessor.transform(X_test_eng)

feature_names = preprocessor.get_feature_names_out()

X_train_encoded = pd.DataFrame(
    X_train_encoded,
    columns=feature_names,
    index=X_train_eng.index
)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=feature_names,
    index=X_test_eng.index
)

print("Encoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)
display(X_train_encoded.head())

Encoded training shape: (2344, 315)
Encoded testing shape: (586, 315)


,numeric__MS SubClass,numeric__Lot Frontage,numeric__Lot Area,numeric__Overall Qual,numeric__Overall Cond,numeric__Mas Vnr Area,numeric__BsmtFin SF 1,numeric__BsmtFin SF 2,numeric__Bsmt Unf SF,numeric__Total Bsmt SF,numeric__1st Flr SF,numeric__2nd Flr SF,numeric__Low Qual Fin SF,numeric__Gr Liv Area,numeric__Bsmt Full Bath,numeric__Bsmt Half Bath,numeric__Full Bath,numeric__Half Bath,numeric__Bedroom AbvGr,numeric__Kitchen AbvGr,numeric__TotRms AbvGrd,numeric__Fireplaces,numeric__Garage Yr Blt,numeric__Garage Cars,numeric__Garage Area,numeric__Wood Deck SF,numeric__Open Porch SF,numeric__Enclosed Porch,numeric__3Ssn Porch,numeric__Screen Porch,numeric__Pool Area,numeric__Misc Val,numeric__Mo Sold,numeric__Yr Sold,numeric__House Age,numeric__Years Since Remodel,categorical__MS Zoning_A (agr),categorical__MS Zoning_C (all),categorical__MS Zoning_FV,categorical__MS Zoning_I (all),categorical__MS Zoning_RH,categorical__MS Zoning_RL,categorical__MS Zoning_RM,categorical__Street_Grvl,categorical__Street_Pave,categorical__Alley_Grvl,categorical__Alley_Not Available,categorical__Alley_Pave,categorical__Lot Shape_IR1,categorical__Lot Shape_IR2,...,categorical__Garage Finish_Fin,categorical__Garage Finish_Not Available,categorical__Garage Finish_RFn,categorical__Garage Finish_Unf,categorical__Garage Qual_Ex,categorical__Garage Qual_Fa,categorical__Garage Qual_Gd,categorical__Garage Qual_Not Available,categorical__Garage Qual_Po,categorical__Garage Qual_TA,categorical__Garage Cond_Ex,categorical__Garage Cond_Fa,categorical__Garage Cond_Gd,categorical__Garage Cond_Not Available,categorical__Garage Cond_Po,categorical__Garage Cond_TA,categorical__Paved Drive_N,categorical__Paved Drive_P,categorical__Paved Drive_Y,categorical__Pool QC_Ex,categorical__Pool QC_Fa,categorical__Pool QC_Gd,categorical__Pool QC_Not Available,categorical__Pool QC_TA,categorical__Fence_GdPrv,categorical__Fence_GdWo,categorical__Fence_MnPrv,categorical__Fence_MnWw,categorical__Fence_Not Available,categorical__Misc Feature_Elev,categorical__Misc Feature_Gar2,categorical__Misc Feature_Not Available,categorical__Misc Feature_Othr,categorical__Misc Feature_Shed,categorical__Sale Type_COD,categorical__Sale Type_CWD,categorical__Sale Type_Con,categorical__Sale Type_ConLD,categorical__Sale Type_ConLI,categorical__Sale Type_ConLw,categorical__Sale Type_New,categorical__Sale Type_Oth,categorical__Sale Type_VWD,categorical__Sale Type_WD,categorical__Sale Condition_Abnorml,categorical__Sale Condition_AdjLand,categorical__Sale Condition_Alloca,categorical__Sale Condition_Family,categorical__Sale Condition_Normal,categorical__Sale Condition_Partial
381,-0.871817,0.514642,0.033810,0.673941,-0.526415,0.531409,-0.978912,-0.293998,1.235322,0.098542,0.558877,-0.781379,-0.098322,-0.243522,-0.820994,-0.245505,0.80217,-0.742207,0.188817,-0.203928,-0.261706,0.614268,-0.059269,0.339058,0.046700,-0.753959,-0.695967,-0.369038,-0.098812,-0.286876,-0.067407,-0.09315,-0.068220,0.916619,-0.141706,0.438854,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
834,0.062906,-0.047047,2.307082,-0.766750,-0.526415,-0.569155,-0.428067,4.191480,-1.278475,-0.077908,0.013468,1.480650,-0.098322,1.254956,-0.820994,3.894526,0.80217,1.259788,0.188817,-0.203928,0.377959,-0.917482,-0.419286,0.339058,0.324553,3.139485,-0.695967,-0.369038,-0.098812,3.744734,-0.067407,-0.09315,-0.068220,0.916619,0.154629,0.872222,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1898,0.763949,0.046568,-0.035514,-1.487095,-0.526415,-0.569155,-0.978912,-0.293998,-1.278475,-2.399267,0.179688,-0.781379,-0.098322,-0.532910,-

## 11. Feature selection using LassoCV

We use **LassoCV** instead of `SelectKBest` to select features for the linear regression workflow.

- Lasso is a linear regression model with an L1 penalty. It can shrink some feature coefficients to exactly zero.
- `LassoCV` tests multiple regularization strengths (`alpha`) using cross-validation on the training data and chooses the best one.
- All encoded features are standardized before fitting Lasso so the penalty is applied on a comparable scale.
- The scaler and Lasso model are fitted using training data only. The test data is transformed with the already-fitted scaler.
- Features with non-zero Lasso coefficients are retained for the next stage.

**Important:** Feature selection is part of model training. Do not use `y_test` to choose features or tune alpha.

In [20]:
# Standardize all encoded columns before applying the L1 penalty.
# Numeric columns were scaled in the preprocessing pipeline, but one-hot columns
# are 0/1 indicators. Scaling the complete encoded matrix gives Lasso a more
# comparable scale across feature types.
encoded_scaler = StandardScaler()

X_train_scaled_array = encoded_scaler.fit_transform(X_train_encoded)
X_test_scaled_array = encoded_scaler.transform(X_test_encoded)

X_train_scaled = pd.DataFrame(
    X_train_scaled_array,
    columns=X_train_encoded.columns,
    index=X_train_encoded.index
)

X_test_scaled = pd.DataFrame(
    X_test_scaled_array,
    columns=X_test_encoded.columns,
    index=X_test_encoded.index
)

# LassoCV selects the regularization strength using cross-validation
# performed only on the training data.
lasso_model = LassoCV(
    cv=5,
    max_iter=20000,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

lasso_model.fit(X_train_scaled, y_train)

print("Best alpha selected by cross-validation:", lasso_model.alpha_)

# Store one coefficient for each encoded feature.
lasso_coefficients = pd.Series(
    lasso_model.coef_,
    index=X_train_scaled.columns,
    name="coefficient"
)

# Keep features whose coefficients are not effectively zero.
COEF_THRESHOLD = 1e-8
selected_feature_names = lasso_coefficients[
    lasso_coefficients.abs() > COEF_THRESHOLD
].index.tolist()

if len(selected_feature_names) == 0:
    raise ValueError(
        "Lasso selected zero features. Review the data and consider adjusting the model setup."
    )

X_train_selected = X_train_scaled[selected_feature_names].copy()
X_test_selected = X_test_scaled[selected_feature_names].copy()

print("Total encoded features:", len(lasso_coefficients))
print("Selected features:", len(selected_feature_names))
print("\nSelected feature names:")
for feature in selected_feature_names:
    print("-", feature)

Best alpha selected by cross-validation: 571.9125650831907
Total encoded features: 315
Selected features: 111

Selected feature names:
- numeric__MS SubClass
- numeric__Lot Frontage
- numeric__Lot Area
- numeric__Overall Qual
- numeric__Overall Cond
- numeric__Mas Vnr Area
- numeric__BsmtFin SF 1
- numeric__BsmtFin SF 2
- numeric__Total Bsmt SF
- numeric__Low Qual Fin SF
- numeric__Gr Liv Area
- numeric__Bsmt Full Bath
- numeric__Bsmt Half Bath
- numeric__Full Bath
- numeric__Bedroom AbvGr
- numeric__Kitchen AbvGr
- numeric__Fireplaces
- numeric__Garage Yr Blt
- numeric__Garage Cars
- numeric__Garage Area
- numeric__Wood Deck SF
- numeric__Screen Porch
- numeric__Yr Sold
- numeric__House Age
- numeric__Years Since Remodel
- categorical__MS Zoning_C (all)
- categorical__MS Zoning_RM
- categorical__Street_Grvl
- categorical__Lot Shape_IR2
- categorical__Land Contour_Bnk
- categorical__Land Contour_HLS
- categorical__Lot Config_CulDSac
- categorical__Land Slope_Mod
- categorical__Land Slo

In [21]:
# Review all Lasso coefficients, ordered by absolute magnitude.
# A coefficient of zero means Lasso did not retain that feature.
coefficient_table = lasso_coefficients.to_frame()
coefficient_table["absolute_coefficient"] = coefficient_table["coefficient"].abs()
coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient", ascending=False
)

display(coefficient_table)

print("\nNumber of features with non-zero coefficients:", len(selected_feature_names))

,coefficient,absolute_coefficient
numeric__Gr Liv Area,27124.809316,27124.809316
categorical__Roof Matl_ClyTile,-11885.485046,11885.485046
numeric__Overall Qual,11134.377820,11134.377820
categorical__Misc Feature_Elev,-9556.106948,9556.106948
numeric__House Age,-8160.252419,8160.252419
...,...,...
categorical__Sale Type_ConLw,0.000000,0.000000
categorical__Sale Type_ConLI,-0.000000,0.000000
categorical__Sale Type_ConLD,0.000000,0.000000
categorical__Sale Condition_Partial,0.000000,0.000000



Number of features with non-zero coefficients: 111


<details>
<summary><strong>Observation notes – Lasso feature selection</strong></summary>

After running the selection cells, record:
- The best alpha selected by `LassoCV`.
- The total number of encoded features before selection.
- The number of features retained by Lasso.
- The features with the largest absolute coefficients.

Lasso uses an L1 penalty to shrink less useful coefficients toward zero. A non-zero coefficient means the feature was retained by this fitted model; it does not prove that the feature causes a change in sale price. Correlated features may compete with one another, so the selected list can change with the training split.
</details>

## 12. Save transformed datasets – selected numeric features only

This is the second requested output. The selected feature columns are numeric because categorical predictors have already been encoded.

We save:
- `ames_train_transformed.csv`
- `ames_test_transformed.csv`
- `ames_transformed.csv` – both splits combined, with a `data_split` label.

The target `SalePrice` and identifiers are included for traceability. `Order` and `PID` are not model features.

In [22]:
# Add target and identifiers back to each split for traceability.
train_output = X_train_selected.copy()
train_output[TARGET] = y_train
train_output["data_split"] = "train"

test_output = X_test_selected.copy()
test_output[TARGET] = y_test
test_output["data_split"] = "test"

for column in id_columns:
    train_output.insert(0, column, ids_train[column])
    test_output.insert(0, column, ids_test[column])

# Save separate train and test files.
TRAIN_PATH = PROCESSED_DIR / "ames_train_transformed.csv"
TEST_PATH = PROCESSED_DIR / "ames_test_transformed.csv"

train_output.to_csv(TRAIN_PATH, index=False)
test_output.to_csv(TEST_PATH, index=False)

# Combine the two splits into one processed file.
combined_output = pd.concat([train_output, test_output], axis=0)
combined_output = combined_output.sort_index()

COMBINED_PATH = PROCESSED_DIR / "ames_transformed.csv"
combined_output.to_csv(COMBINED_PATH, index=False)

print("Training file:", TRAIN_PATH)
print("Testing file:", TEST_PATH)
print("Combined transformed file:", COMBINED_PATH)
print("Combined shape:", combined_output.shape)


Training file: /content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model/data/processed/ames_train_transformed.csv
Testing file: /content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model/data/processed/ames_test_transformed.csv
Combined transformed file: /content/Applied-Machine-Learning/SupervisedLearning/LinearRegression/UsingScikit-Learn/03_Regression/House_Price_Prediction_Model/data/processed/ames_transformed.csv
Combined shape: (2930, 115)


In [23]:
# Verify the saved processed files.
train_check = pd.read_csv(TRAIN_PATH)
test_check = pd.read_csv(TEST_PATH)
processed_check = pd.read_csv(COMBINED_PATH)

print("Train file shape:", train_check.shape)
print("Test file shape:", test_check.shape)
print("Combined file shape:", processed_check.shape)

print("\nSplit counts:")
display(processed_check["data_split"].value_counts())

print("\nPreview of transformed data:")
display(processed_check.head())

Train file shape: (2344, 115)
Test file shape: (586, 115)
Combined file shape: (2930, 115)

Split counts:


,count
data_split,
train,2344
test,586



Preview of transformed data:


,PID,Order,numeric__MS SubClass,numeric__Lot Frontage,numeric__Lot Area,numeric__Overall Qual,numeric__Overall Cond,numeric__Mas Vnr Area,numeric__BsmtFin SF 1,numeric__BsmtFin SF 2,numeric__Total Bsmt SF,numeric__Low Qual Fin SF,numeric__Gr Liv Area,numeric__Bsmt Full Bath,numeric__Bsmt Half Bath,numeric__Full Bath,numeric__Bedroom AbvGr,numeric__Kitchen AbvGr,numeric__Fireplaces,numeric__Garage Yr Blt,numeric__Garage Cars,numeric__Garage Area,numeric__Wood Deck SF,numeric__Screen Porch,numeric__Yr Sold,numeric__House Age,numeric__Years Since Remodel,categorical__MS Zoning_C (all),categorical__MS Zoning_RM,categorical__Street_Grvl,categorical__Lot Shape_IR2,categorical__Land Contour_Bnk,categorical__Land Contour_HLS,categorical__Lot Config_CulDSac,categorical__Land Slope_Mod,categorical__Land Slope_Sev,categorical__Neighborhood_BrkSide,categorical__Neighborhood_Crawfor,categorical__Neighborhood_GrnHill,categorical__Neighborhood_Mitchel,categorical__Neighborhood_NAmes,categorical__Neighborhood_NPkVill,categorical__Neighborhood_NWAmes,categorical__Neighborhood_NoRidge,categorical__Neighborhood_NridgHt,categorical__Neighborhood_Somerst,categorical__Neighborhood_StoneBr,categorical__Condition 1_Artery,categorical__Condition 1_Feedr,categorical__Condition 1_Norm,...,categorical__Exterior 2nd_Plywood,categorical__Exterior 2nd_PreCast,categorical__Mas Vnr Type_BrkCmn,categorical__Mas Vnr Type_BrkFace,categorical__Mas Vnr Type_CBlock,categorical__Mas Vnr Type_Stone,categorical__Exter Qual_Ex,categorical__Exter Qual_TA,categorical__Foundation_CBlock,categorical__Foundation_PConc,categorical__Foundation_Wood,categorical__Bsmt Qual_Ex,categorical__Bsmt Exposure_Gd,categorical__Bsmt Exposure_Mn,categorical__Bsmt Exposure_No,categorical__BsmtFin Type 1_GLQ,categorical__BsmtFin Type 1_LwQ,categorical__BsmtFin Type 1_Rec,categorical__BsmtFin Type 2_ALQ,categorical__BsmtFin Type 2_GLQ,categorical__Heating_OthW,categorical__Heating QC_Ex,categorical__Heating QC_TA,categorical__Kitchen Qual_Ex,categorical__Kitchen Qual_TA,categorical__Functional_Maj1,categorical__Functional_Sal,categorical__Functional_Sev,categorical__Functional_Typ,categorical__Fireplace Qu_Ex,categorical__Garage Type_2Types,categorical__Garage Type_Basment,categorical__Garage Type_BuiltIn,categorical__Garage Type_Not Available,categorical__Garage Finish_Fin,categorical__Garage Finish_RFn,categorical__Garage Qual_Ex,categorical__Garage Qual_Gd,categorical__Garage Cond_Ex,categorical__Garage Cond_Fa,categorical__Pool QC_Ex,categorical__Pool QC_Not Available,categorical__Misc Feature_Elev,categorical__Misc Feature_Othr,categorical__Sale Type_Con,categorical__Sale Type_New,categorical__Sale Condition_Abnorml,categorical__Sale Condition_Family,SalePrice,data_split
0,526301100,1,-0.871817,3.369893,2.688735,-0.046404,-0.526415,0.083031,0.434702,-0.293998,0.075627,-0.098322,0.323363,1.082477,-0.245505,-1.009281,0.188817,-0.203928,2.146018,-0.699299,0.339058,0.277460,0.928394,-0.286876,1.67717,0.418038,1.257439,-0.097338,-0.433634,-0.068666,-0.152099,-0.198665,-0.207771,-0.249149,-0.213296,-0.077515,-0.19279,-0.198665,-0.029223,-0.207771,2.320974,-0.080253,-0.220838,-0.157872,-0.233304,-0.248182,-0.124892,-0.179289,-0.244284,0.404074,...,3.068840,-0.020659,-0.095079,-0.661843,-0.020659,3.44661,-0.190397,0.777419,1.140305,-0.876198,-0.046235,-0.294439,3.099409,-0.302072,-1.385299,-0.637274,-0.242316,-0.330641,-0.133427,-0.107949,-0.029223,-1.008569,-0.640585,-0.261473,0.956573,-0.077515,-0.020659,-0.029223,0.271573,-0.123118,-0.087969,-0.113862,-0.250113,-0.232286,1.762123,-0.622079,-0.035798,-0.087969,-0.035798,-0.162079,-0.041345,0.071734,-0.020659,-0.041345,-0.046235,-0.281407,-0.268846,-0.133427,215000,train
1,526350040,2,-0.871817,0.514642,0.185626,-0.766750,0.378216,-0.569155,0.056411,0.551432,-0.378104,-0.098322,-1.183043,-0.820994,-0.245505,-1.009281,-1.030432,-0.203928,-0.917482,-0.659297,-1.001165,1.228754,0.367610,1.863316,1.67717,0.385112,1.209287,-0.097338,-0.433634,-0.06

In [24]:
# Confirm that selected predictor columns are numeric.
selected_columns_check = selected_feature_names

print("All selected predictors are numeric:",
      all(pd.api.types.is_numeric_dtype(processed_check[column])
          for column in selected_columns_check))

print("Missing values in selected predictors:",
      processed_check[selected_columns_check].isna().sum().sum())


All selected predictors are numeric: True
Missing values in selected predictors: 0




---





```

```





---



**Older Version code**



```
Just for reference
```



In [ ]:
df = df.drop(["Order","PID"],axis=1)
print(df.shape)

(2930, 80)


In [ ]:
X = df.drop(['SalePrice'],axis=1)
y = df['SalePrice']

In [ ]:
print(X.head())
print(y.head())

   MS SubClass MS Zoning  Lot Frontage  Lot Area Street Alley Lot Shape  \
0           20        RL         141.0     31770   Pave   NaN       IR1   
1           20        RH          80.0     11622   Pave   NaN       Reg   
2           20        RL          81.0     14267   Pave   NaN       IR1   
3           20        RL          93.0     11160   Pave   NaN       Reg   
4           60        RL          74.0     13830   Pave   NaN       IR1   

  Land Contour Utilities Lot Config Land Slope Neighborhood Condition 1  \
0          Lvl    AllPub     Corner        Gtl        NAmes        Norm   
1          Lvl    AllPub     Inside        Gtl        NAmes       Feedr   
2          Lvl    AllPub     Corner        Gtl        NAmes        Norm   
3          Lvl    AllPub     Corner        Gtl        NAmes        Norm   
4          Lvl    AllPub     Inside        Gtl      Gilbert        Norm   

  Condition 2 Bldg Type House Style  Overall Qual  Overall Cond  Year Built  \
0        Norm      

**Split the Data in Train and Test**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
print(X_train.shape)
print(y_train.shape)

(2344, 79)
(2344,)


**Handling NaN/Null Value**

1) Categorical Columns

In [ ]:
Categoricalf_na = [feature for feature in X_train.columns if X_train[feature].isnull().sum()>1 and X_train[feature].dtypes == 'O' ]

In [ ]:
X_train[Categoricalf_na].head()

,Alley,Mas Vnr Type,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Pool QC,Fence,Misc Feature
381,NaN,BrkFace,Gd,TA,No,Unf,Unf,TA,Attchd,RFn,TA,TA,NaN,MnPrv,NaN
834,NaN,NaN,TA,Gd,Gd,LwQ,ALQ,NaN,Attchd,Fin,TA,TA,NaN,NaN,NaN
1898,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CarPort,Unf,TA,TA,NaN,NaN,NaN
678,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Detchd,Unf,TA,TA,NaN,NaN,NaN
700,NaN,NaN,Fa,Po,No,Unf,Unf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
X_test[Categoricalf_na].head()

,Alley,Mas Vnr Type,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Pool QC,Fence,Misc Feature
1357,Grvl,NaN,TA,TA,No,Unf,Unf,Gd,Detchd,Unf,TA,TA,NaN,MnPrv,NaN
2367,NaN,BrkFace,TA,TA,No,ALQ,Unf,NaN,Detchd,Unf,TA,TA,NaN,NaN,NaN
2822,NaN,BrkFace,Gd,TA,No,Unf,Unf,Gd,BuiltIn,Fin,TA,TA,NaN,NaN,NaN
2126,NaN,NaN,Gd,TA,No,GLQ,Unf,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1544,NaN,NaN,TA,TA,No,Rec,GLQ,NaN,Detchd,Unf,TA,TA,NaN,MnPrv,NaN


In [ ]:
# missing feature with percent of missing values

for feature in Categoricalf_na:
  print(f"Feature:{feature} and missing Value Percent:{np.round(X_train[feature].isnull().mean(),4)}%")

Feature:Alley and missing Value Percent:0.9309%
Feature:Mas Vnr Type and missing Value Percent:0.6084%
Feature:Bsmt Qual and missing Value Percent:0.026%
Feature:Bsmt Cond and missing Value Percent:0.026%
Feature:Bsmt Exposure and missing Value Percent:0.0269%
Feature:BsmtFin Type 1 and missing Value Percent:0.026%
Feature:BsmtFin Type 2 and missing Value Percent:0.0265%
Feature:Fireplace Qu and missing Value Percent:0.4881%
Feature:Garage Type and missing Value Percent:0.0512%
Feature:Garage Finish and missing Value Percent:0.052%
Feature:Garage Qual and missing Value Percent:0.052%
Feature:Garage Cond and missing Value Percent:0.052%
Feature:Pool QC and missing Value Percent:0.9949%
Feature:Fence and missing Value Percent:0.7995%
Feature:Misc Feature and missing Value Percent:0.9599%


In [ ]:
Categoricalf_na = [feature for feature in X_train.columns if X_train[feature].isnull().sum()>1 and X_train[feature].dtypes == 'O' ]

In [ ]:
for feature in Categoricalf_na:
  print(f"Feature:{feature} and missing Value Percent:{np.round(X_test[feature].isnull().mean(),4)}%")

Feature:Alley and missing Value Percent:0.9386%
Feature:Mas Vnr Type and missing Value Percent:0.5956%
Feature:Bsmt Qual and missing Value Percent:0.0324%
Feature:Bsmt Cond and missing Value Percent:0.0324%
Feature:Bsmt Exposure and missing Value Percent:0.0341%
Feature:BsmtFin Type 1 and missing Value Percent:0.0324%
Feature:BsmtFin Type 2 and missing Value Percent:0.0324%
Feature:Fireplace Qu and missing Value Percent:0.4744%
Feature:Garage Type and missing Value Percent:0.0631%
Feature:Garage Finish and missing Value Percent:0.0631%
Feature:Garage Qual and missing Value Percent:0.0631%
Feature:Garage Cond and missing Value Percent:0.0631%
Feature:Pool QC and missing Value Percent:0.9983%
Feature:Fence and missing Value Percent:0.8259%
Feature:Misc Feature and missing Value Percent:0.9795%


**Creating new label as Missing For NaN/Null values**

In [ ]:
def replace_cat_feature(X_train,Categoricalf_na):
  data = X_train.copy()

  data[Categoricalf_na] = data[Categoricalf_na].fillna("Missing")
  return data



X_train = replace_cat_feature(X_train,Categoricalf_na)

# check categorical feature have Null value or not

X_train[Categoricalf_na].isnull().sum()

,0
Alley,0
Mas Vnr Type,0
Bsmt Qual,0
Bsmt Cond,0
Bsmt Exposure,0
BsmtFin Type 1,0
BsmtFin Type 2,0
Fireplace Qu,0
Garage Type,0
Garage Finish,0


In [ ]:
X_train.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition
381,20,RL,80.0,10400,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,Norm,1Fam,1Story,7,5,1976,1976,Gable,CompShg,HdBoard,HdBoard,BrkFace,189.0,TA,TA,CBlock,Gd,TA,No,Unf,0.0,Unf,0.0,1090.0,1090.0,GasA,TA,Y,SBrkr,1370,0,0,1370,0.0,0.0,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2.0,479.0,TA,TA,Y,0,0,0,0,0,0,Missing,MnPrv,Missing,0,6,2009,WD,Family
834,60,RL,NaN,28698,Pave,Missing,IR2,Low,AllPub,CulDSac,Sev,ClearCr,Norm,Norm,1Fam,2Story,5,5,1967,1967,Flat,Tar&Grv,Plywood,Plywood,Missing,0.0,TA,TA,PConc,TA,Gd,Gd,LwQ,249.0,ALQ,764.0,0.0,1013.0,GasA,TA,Y,SBrkr,1160,966,0,2126,0.0,1.0,2,1,3,1,TA,7,Min2,0,Missing,Attchd,1967.0,Fin,2.0,538.0,TA,TA,Y,486,0,0,0,225,0,Missing,Missing,Missing,0,6,2009,WD,Abnorml
1898,90,RL,70.0,9842,Pave,Missing,Reg,Lvl,AllPub,FR2,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,1962,1962,Gable,CompShg,HdBoard,HdBoard,Missing,0.0,TA,TA,Slab,Missing,Missing,Missing,Missing,0.0,Missing,0.0,0.0,0.0,GasA,TA,Y,SBrkr,1224,0,0,1224,0.0,0.0,2,0,2,2,TA,6,Typ,0,Missing,CarPort,1962.0,Unf,2.0,462.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,3,2007,WD,Normal
678,90,RL,60.0,7200,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,1949,1950,Gable,CompShg,BrkFace,Stone,Missing,0.0,TA,TA,Slab,Missing,Missing,Missing,Missing,0.0,Missing,0.0,0.0,0.0,Wall,Fa,N,FuseF,1040,0,0,1040,0.0,0.0,2,0,2,2,TA,6,Typ,0,Missing,Detchd,1956.0,Unf,2.0,420.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,6,2009,WD,Normal
700,190,RM,63.0,7627,Pave,Missing,Reg,Lvl,AllPub,Corner,Gtl,OldTown,Artery,Norm,2fmCon,2Story,4,6,1920,1950,Gable,CompShg,AsbShng,AsbShng,Missing,0.0,Fa,TA,BrkTil,Fa,Po,No,Unf,0.0,Unf,0.0,600.0,600.0,GasA,Gd,N,SBrkr,1101,600,0,1701,0.0,0.0,2,0,4,2,Fa,8,Typ,0,Missing,Missing,NaN,Missing,0.0,0.0,Missing,Missing,N,0,0,148,0,0,0,Missing,Missing,Missing,0,10,2009,WD,Normal


In [ ]:
def replace_cat_feature(X_test,Categoricalf_na):
  data = X_test.copy()

  data[Categoricalf_na] = data[Categoricalf_na].fillna("Missing")
  return data



X_test = replace_cat_feature(X_test,Categoricalf_na)

# check categorical feature have Null value or not

X_test[Categoricalf_na].isnull().sum()

,0
Alley,0
Mas Vnr Type,0
Bsmt Qual,0
Bsmt Cond,0
Bsmt Exposure,0
BsmtFin Type 1,0
BsmtFin Type 2,0
Fireplace Qu,0
Garage Type,0
Garage Finish,0


In [ ]:
X_test.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition
1357,70,RM,NaN,5100,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Norm,Norm,1Fam,2Story,8,7,1925,1996,Hip,CompShg,Stucco,Wd Shng,Missing,0.0,TA,Gd,PConc,TA,TA,No,Unf,0.0,Unf,0.0,588.0,588.0,GasA,Fa,Y,SBrkr,833,833,0,1666,0.0,0.0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1925.0,Unf,1.0,228.0,TA,TA,Y,192,63,0,0,0,0,Missing,MnPrv,Missing,0,6,2008,WD,Normal
2367,160,RM,21.0,1890,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,BrDale,Norm,Norm,Twnhs,2Story,6,7,1972,1972,Gable,CompShg,HdBoard,HdBoard,BrkFace,380.0,TA,TA,CBlock,TA,TA,No,ALQ,282.0,Unf,0.0,212.0,494.0,GasA,Ex,Y,SBrkr,494,536,0,1030,0.0,0.0,1,1,3,1,TA,6,Typ,0,Missing,Detchd,1973.0,Unf,1.0,264.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,7,2006,WD,Normal
2822,60,RL,62.0,7162,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Norm,Norm,1Fam,2Story,7,5,2003,2004,Hip,CompShg,HdBoard,Stucco,BrkFace,190.0,Gd,TA,PConc,Gd,TA,No,Unf,0.0,Unf,0.0,796.0,796.0,GasA,Ex,Y,SBrkr,806,918,0,1724,0.0,0.0,2,1,3,1,Gd,8,Typ,1,Gd,BuiltIn,2003.0,Fin,2.0,616.0,TA,TA,Y,168,57,0,0,0,0,Missing,Missing,Missing,0,5,2006,WD,Normal
2126,20,RL,60.0,8070,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,1Story,4,5,1994,1995,Gable,CompShg,VinylSd,VinylSd,Missing,0.0,TA,TA,PConc,Gd,TA,No,GLQ,588.0,Unf,0.0,402.0,990.0,GasA,Ex,Y,SBrkr,990,0,0,990,1.0,0.0,1,0,3,1,TA,5,Typ,0,Missing,Missing,NaN,Missing,0.0,0.0,Missing,Missing,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,8,2007,WD,Normal
1544,30,RM,50.0,7000,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,IDOTRR,Norm,Norm,1Fam,1Story,6,8,1926,1998,Gable,CompShg,VinylSd,VinylSd,Missing,0.0,TA,Gd,BrkTil,TA,TA,No,Rec,299.0,GLQ,40.0,555.0,894.0,GasA,TA,Y,SBrkr,919,0,0,919,1.0,0.0,1,0,2,1,TA,5,Typ,0,Missing,Detchd,1926.0,Unf,1.0,195.0,TA,TA,P,0,0,116,0,0,0,Missing,MnPrv,Missing,0,7,2008,WD,Normal


**Numerical Features that Contains NaN/Null Values**

In [ ]:
numericalf_na = [feature for feature in X_train.columns if X_train[feature].isnull().sum()>1 and X_train[feature].dtypes !='O']

In [ ]:
X_train[numericalf_na].head()

,Lot Frontage,Mas Vnr Area,Garage Yr Blt
381,80.0,189.0,1976.0
834,NaN,0.0,1967.0
1898,70.0,0.0,1962.0
678,60.0,0.0,1956.0
700,63.0,0.0,NaN


In [ ]:
# finding each feature with missing value percent

for feature in numericalf_na:
  print(f"Feature:{feature} and Missing Value in Percent:{np.round(X_train[feature].isnull().mean(),4)}%")

Feature:Lot Frontage and Missing Value in Percent:0.1677%
Feature:Mas Vnr Area and Missing Value in Percent:0.0081%
Feature:Garage Yr Blt and Missing Value in Percent:0.052%


In [ ]:
# Replacing Numerical NaN values as median with creating new feature featue+'nan'

for feature in numericalf_na:
  median_value = X_train[feature].median()

  X_train[feature+'NaN'] = np.where(X_train[feature].isnull(),1,0)
  X_train[feature].fillna(median_value,inplace=True) # inplace = True that means change to the Original Table


X_train[numericalf_na].isnull().sum()


/tmp/ipykernel_508/3693465190.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_train[feature].fillna(median_value,inplace=True) # inplace = True that means change to the Original Table
/tmp/ipykernel_508/3693465190.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exampl

,0
Lot Frontage,0
Mas Vnr Area,0
Garage Yr Blt,0


In [ ]:
X_train.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Yr Blt,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,Lot FrontageNaN,Mas Vnr AreaNaN,Garage Yr BltNaN
381,20,RL,80.0,10400,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,Norm,1Fam,1Story,7,5,1976,1976,Gable,CompShg,HdBoard,HdBoard,BrkFace,189.0,TA,TA,CBlock,Gd,TA,No,Unf,0.0,Unf,0.0,1090.0,1090.0,GasA,TA,Y,SBrkr,1370,0,0,1370,0.0,0.0,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2.0,479.0,TA,TA,Y,0,0,0,0,0,0,Missing,MnPrv,Missing,0,6,2009,WD,Family,0,0,0
834,60,RL,68.0,28698,Pave,Missing,IR2,Low,AllPub,CulDSac,Sev,ClearCr,Norm,Norm,1Fam,2Story,5,5,1967,1967,Flat,Tar&Grv,Plywood,Plywood,Missing,0.0,TA,TA,PConc,TA,Gd,Gd,LwQ,249.0,ALQ,764.0,0.0,1013.0,GasA,TA,Y,SBrkr,1160,966,0,2126,0.0,1.0,2,1,3,1,TA,7,Min2,0,Missing,Attchd,1967.0,Fin,2.0,538.0,TA,TA,Y,486,0,0,0,225,0,Missing,Missing,Missing,0,6,2009,WD,Abnorml,1,0,0
1898,90,RL,70.0,9842,Pave,Missing,Reg,Lvl,AllPub,FR2,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,1962,1962,Gable,CompShg,HdBoard,HdBoard,Missing,0.0,TA,TA,Slab,Missing,Missing,Missing,Missing,0.0,Missing,0.0,0.0,0.0,GasA,TA,Y,SBrkr,1224,0,0,1224,0.0,0.0,2,0,2,2,TA,6,Typ,0,Missing,CarPort,1962.0,Unf,2.0,462.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,3,2007,WD,Normal,0,0,0
678,90,RL,60.0,7200,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,1949,1950,Gable,CompShg,BrkFace,Stone,Missing,0.0,TA,TA,Slab,Missing,Missing,Missing,Missing,0.0,Missing,0.0,0.0,0.0,Wall,Fa,N,FuseF,1040,0,0,1040,0.0,0.0,2,0,2,2,TA,6,Typ,0,Missing,Detchd,1956.0,Unf,2.0,420.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,6,2009,WD,Normal,0,0,0
700,190,RM,63.0,7627,Pave,Missing,Reg,Lvl,AllPub,Corner,Gtl,OldTown,Artery,Norm,2fmCon,2Story,4,6,1920,1950,Gable,CompShg,AsbShng,AsbShng,Missing,0.0,Fa,TA,BrkTil,Fa,Po,No,Unf,0.0,Unf,0.0,600.0,600.0,GasA,Gd,N,SBrkr,1101,600,0,1701,0.0,0.0,2,0,4,2,Fa,8,Typ,0,Missing,Missing,1978.0,Missing,0.0,0.0,Missing,Missing,N,0,0,148,0,0,0,Missing,Missing,Missing,0,10,2009,WD,Normal,0,0,1


In [ ]:
for feature in numericalf_na:
  print(f"Feature:{feature} and Missing Value in Percent:{np.round(X_test[feature].isnull().mean(),4)}%")

Feature:Lot Frontage and Missing Value in Percent:0.1655%
Feature:Mas Vnr Area and Missing Value in Percent:0.0068%
Feature:Garage Yr Blt and Missing Value in Percent:0.0631%


In [ ]:
# Replacing Numerical NaN values as median with creating new feature featue+'nan'

for feature in numericalf_na:
  median_value = X_test[feature].median()

  X_test[feature+'NaN'] = np.where(X_test[feature].isnull(),1,0)
  X_test[feature].fillna(median_value,inplace=True) # inplace = True that means change to the Original Table


X_test[numericalf_na].isnull().sum()

/tmp/ipykernel_508/2374706135.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  X_test[feature].fillna(median_value,inplace=True) # inplace = True that means change to the Original Table
/tmp/ipykernel_508/2374706135.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example

,0
Lot Frontage,0
Mas Vnr Area,0
Garage Yr Blt,0


In [ ]:
for feature in ['Year Built', 'Year Remod/Add', 'Garage Yr Blt']:
    X_train[feature] = X_train['Yr Sold'] - X_train[feature]

In [ ]:
X_train = X_train.rename(columns={
    'Year Built':     'House Age',
    'Year Remod/Add': 'Years Since Remodel',
    'Garage Yr Blt':  'Garage Age',
})

In [ ]:
for feature in ['Year Built', 'Year Remod/Add', 'Garage Yr Blt']:
    X_test[feature] = X_test['Yr Sold'] - X_test[feature]

X_test = X_test.rename(columns={
    'Year Built':     'House Age',
    'Year Remod/Add': 'Years Since Remodel',
    'Garage Yr Blt':  'Garage Age',
})

In [ ]:
X_train.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,House Age,Years Since Remodel,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Age,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,Lot FrontageNaN,Mas Vnr AreaNaN,Garage Yr BltNaN
381,20,RL,80.0,10400,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,Norm,1Fam,1Story,7,5,33,33,Gable,CompShg,HdBoard,HdBoard,BrkFace,189.0,TA,TA,CBlock,Gd,TA,No,Unf,0.0,Unf,0.0,1090.0,1090.0,GasA,TA,Y,SBrkr,1370,0,0,1370,0.0,0.0,2,0,3,1,TA,6,Typ,1,TA,Attchd,33.0,RFn,2.0,479.0,TA,TA,Y,0,0,0,0,0,0,Missing,MnPrv,Missing,0,6,2009,WD,Family,0,0,0
834,60,RL,68.0,28698,Pave,Missing,IR2,Low,AllPub,CulDSac,Sev,ClearCr,Norm,Norm,1Fam,2Story,5,5,42,42,Flat,Tar&Grv,Plywood,Plywood,Missing,0.0,TA,TA,PConc,TA,Gd,Gd,LwQ,249.0,ALQ,764.0,0.0,1013.0,GasA,TA,Y,SBrkr,1160,966,0,2126,0.0,1.0,2,1,3,1,TA,7,Min2,0,Missing,Attchd,42.0,Fin,2.0,538.0,TA,TA,Y,486,0,0,0,225,0,Missing,Missing,Missing,0,6,2009,WD,Abnorml,1,0,0
1898,90,RL,70.0,9842,Pave,Missing,Reg,Lvl,AllPub,FR2,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,45,45,Gable,CompShg,HdBoard,HdBoard,Missing,0.0,TA,TA,Slab,Missing,Missing,Missing,Missing,0.0,Missing,0.0,0.0,0.0,GasA,TA,Y,SBrkr,1224,0,0,1224,0.0,0.0,2,0,2,2,TA,6,Typ,0,Missing,CarPort,45.0,Unf,2.0,462.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,3,2007,WD,Normal,0,0,0
678,90,RL,60.0,7200,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,Duplex,1Story,4,5,60,59,Gable,CompShg,BrkFace,Stone,Missing,0.0,TA,TA,Slab,Missing,Missing,Missing,Missing,0.0,Missing,0.0,0.0,0.0,Wall,Fa,N,FuseF,1040,0,0,1040,0.0,0.0,2,0,2,2,TA,6,Typ,0,Missing,Detchd,53.0,Unf,2.0,420.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,6,2009,WD,Normal,0,0,0
700,190,RM,63.0,7627,Pave,Missing,Reg,Lvl,AllPub,Corner,Gtl,OldTown,Artery,Norm,2fmCon,2Story,4,6,89,59,Gable,CompShg,AsbShng,AsbShng,Missing,0.0,Fa,TA,BrkTil,Fa,Po,No,Unf,0.0,Unf,0.0,600.0,600.0,GasA,Gd,N,SBrkr,1101,600,0,1701,0.0,0.0,2,0,4,2,Fa,8,Typ,0,Missing,Missing,31.0,Missing,0.0,0.0,Missing,Missing,N,0,0,148,0,0,0,Missing,Missing,Missing,0,10,2009,WD,Normal,0,0,1


In [ ]:
X_test.head()

,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Overall Qual,Overall Cond,House Age,Years Since Remodel,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Mas Vnr Area,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin SF 1,BsmtFin Type 2,BsmtFin SF 2,Bsmt Unf SF,Total Bsmt SF,Heating,Heating QC,Central Air,Electrical,1st Flr SF,2nd Flr SF,Low Qual Fin SF,Gr Liv Area,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath,Bedroom AbvGr,Kitchen AbvGr,Kitchen Qual,TotRms AbvGrd,Functional,Fireplaces,Fireplace Qu,Garage Type,Garage Age,Garage Finish,Garage Cars,Garage Area,Garage Qual,Garage Cond,Paved Drive,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,Lot FrontageNaN,Mas Vnr AreaNaN,Garage Yr BltNaN
1357,70,RM,69.0,5100,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Norm,Norm,1Fam,2Story,8,7,83,12,Hip,CompShg,Stucco,Wd Shng,Missing,0.0,TA,Gd,PConc,TA,TA,No,Unf,0.0,Unf,0.0,588.0,588.0,GasA,Fa,Y,SBrkr,833,833,0,1666,0.0,0.0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,83.0,Unf,1.0,228.0,TA,TA,Y,192,63,0,0,0,0,Missing,MnPrv,Missing,0,6,2008,WD,Normal,1,0,0
2367,160,RM,21.0,1890,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,BrDale,Norm,Norm,Twnhs,2Story,6,7,34,34,Gable,CompShg,HdBoard,HdBoard,BrkFace,380.0,TA,TA,CBlock,TA,TA,No,ALQ,282.0,Unf,0.0,212.0,494.0,GasA,Ex,Y,SBrkr,494,536,0,1030,0.0,0.0,1,1,3,1,TA,6,Typ,0,Missing,Detchd,33.0,Unf,1.0,264.0,TA,TA,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,7,2006,WD,Normal,0,0,0
2822,60,RL,62.0,7162,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Norm,Norm,1Fam,2Story,7,5,3,2,Hip,CompShg,HdBoard,Stucco,BrkFace,190.0,Gd,TA,PConc,Gd,TA,No,Unf,0.0,Unf,0.0,796.0,796.0,GasA,Ex,Y,SBrkr,806,918,0,1724,0.0,0.0,2,1,3,1,Gd,8,Typ,1,Gd,BuiltIn,3.0,Fin,2.0,616.0,TA,TA,Y,168,57,0,0,0,0,Missing,Missing,Missing,0,5,2006,WD,Normal,0,0,0
2126,20,RL,60.0,8070,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,1Story,4,5,13,12,Gable,CompShg,VinylSd,VinylSd,Missing,0.0,TA,TA,PConc,Gd,TA,No,GLQ,588.0,Unf,0.0,402.0,990.0,GasA,Ex,Y,SBrkr,990,0,0,990,1.0,0.0,1,0,3,1,TA,5,Typ,0,Missing,Missing,18.0,Missing,0.0,0.0,Missing,Missing,Y,0,0,0,0,0,0,Missing,Missing,Missing,0,8,2007,WD,Normal,0,0,1
1544,30,RM,50.0,7000,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,IDOTRR,Norm,Norm,1Fam,1Story,6,8,82,10,Gable,CompShg,VinylSd,VinylSd,Missing,0.0,TA,Gd,BrkTil,TA,TA,No,Rec,299.0,GLQ,40.0,555.0,894.0,GasA,TA,Y,SBrkr,919,0,0,919,1.0,0.0,1,0,2,1,TA,5,Typ,0,Missing,Detchd,82.0,Unf,1.0,195.0,TA,TA,P,0,0,116,0,0,0,Missing,MnPrv,Missing,0,7,2008,WD,Normal,0,0,0


In [ ]:
# we notice in Histogram feature vs SalePrice
# that Some of Numerical Features are Skewed

# here we convert into normal distribution using log Distribution

skewed_numf = ['Lot Frontage','Lot Area','1st Flr SF','Gr Liv Area']

for feature in skewed_numf:
  X_train[feature] = np.log(X_train[feature])


for feature in skewed_numf:
  X_test[feature] = np.log(X_test[feature])


In [ ]:
y_train_log = np.log(y_train)
y_train_log.head()

y_test_log = np.log(y_test)
y_test_log.head()

,SalePrice
1357,2.484003
2367,2.456280
2822,2.500486
2126,2.461707
1544,2.463346


In [ ]:
# Rare Categorical Feature means Category having less than 1 percent

categorical_f = [feature for feature in X_train.columns if X_train[feature].dtypes=='O']


In [ ]:
rare_labels = {}   # must be created before the loop

for feature in categorical_f:
    freq = X_train[feature].value_counts(normalize=True)   # share of each category
    rare_labels[feature] = freq[freq < 0.01].index.tolist()  # categories under 1%

In [ ]:
for feature in categorical_f:
    X_train[feature] = X_train[feature].where(~X_train[feature].isin(rare_labels[feature]), 'Rare')
    X_test[feature]  = X_test[feature].where(~X_test[feature].isin(rare_labels[feature]), 'Rare')

In [ ]:
X_train[categorical_f].head()

,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,Heating,Heating QC,Central Air,Electrical,Kitchen Qual,Functional,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
381,RL,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NWAmes,Norm,Norm,1Fam,1Story,Gable,CompShg,HdBoard,HdBoard,BrkFace,TA,TA,CBlock,Gd,TA,No,Unf,Unf,GasA,TA,Y,SBrkr,TA,Typ,TA,Attchd,RFn,TA,TA,Y,Missing,MnPrv,Missing,WD,Family
834,RL,Pave,Missing,IR2,Low,AllPub,CulDSac,Rare,ClearCr,Norm,Norm,1Fam,2Story,Rare,Rare,Plywood,Plywood,Missing,TA,TA,PConc,TA,Gd,Gd,LwQ,ALQ,GasA,TA,Y,SBrkr,TA,Min2,Missing,Attchd,Fin,TA,TA,Y,Missing,Missing,Missing,WD,Abnorml
1898,RL,Pave,Missing,Reg,Lvl,AllPub,FR2,Gtl,NAmes,Norm,Norm,Duplex,1Story,Gable,CompShg,HdBoard,HdBoard,Missing,TA,TA,Slab,Missing,Missing,Missing,Missing,Missing,GasA,TA,Y,SBrkr,TA,Typ,Missing,Rare,Unf,TA,TA,Y,Missing,Missing,Missing,WD,Normal
678,RL,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,NAmes,Norm,Norm,Duplex,1Story,Gable,CompShg,BrkFace,Rare,Missing,TA,TA,Slab,Missing,Missing,Missing,Missing,Missing,Rare,Fa,N,FuseF,TA,Typ,Missing,Detchd,Unf,TA,TA,Y,Missing,Missing,Missing,WD,Normal
700,RM,Pave,Missing,Reg,Lvl,AllPub,Corner,Gtl,OldTown,Artery,Norm,2fmCon,2Story,Gable,CompShg,AsbShng,AsbShng,Missing,Fa,TA,BrkTil,Fa,Rare,No,Unf,Unf,GasA,Gd,N,SBrkr,Fa,Typ,Missing,Missing,Missing,Missing,Missing,N,Missing,Missing,Missing,WD,Normal


In [ ]:
X_test[categorical_f].head(10)

,MS Zoning,Street,Alley,Lot Shape,Land Contour,Utilities,Lot Config,Land Slope,Neighborhood,Condition 1,Condition 2,Bldg Type,House Style,Roof Style,Roof Matl,Exterior 1st,Exterior 2nd,Mas Vnr Type,Exter Qual,Exter Cond,Foundation,Bsmt Qual,Bsmt Cond,Bsmt Exposure,BsmtFin Type 1,BsmtFin Type 2,Heating,Heating QC,Central Air,Electrical,Kitchen Qual,Functional,Fireplace Qu,Garage Type,Garage Finish,Garage Qual,Garage Cond,Paved Drive,Pool QC,Fence,Misc Feature,Sale Type,Sale Condition
1357,RM,Pave,Grvl,Reg,Lvl,AllPub,Inside,Gtl,OldTown,Norm,Norm,1Fam,2Story,Hip,CompShg,Stucco,Wd Shng,Missing,TA,Gd,PConc,TA,TA,No,Unf,Unf,GasA,Fa,Y,SBrkr,Gd,Typ,Gd,Detchd,Unf,TA,TA,Y,Missing,MnPrv,Missing,WD,Normal
2367,RM,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,BrDale,Norm,Norm,Twnhs,2Story,Gable,CompShg,HdBoard,HdBoard,BrkFace,TA,TA,CBlock,TA,TA,No,ALQ,Unf,GasA,Ex,Y,SBrkr,TA,Typ,Missing,Detchd,Unf,TA,TA,Y,Missing,Missing,Missing,WD,Normal
2822,RL,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,Edwards,Norm,Norm,1Fam,2Story,Hip,CompShg,HdBoard,Stucco,BrkFace,Gd,TA,PConc,Gd,TA,No,Unf,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,Gd,BuiltIn,Fin,TA,TA,Y,Missing,Missing,Missing,WD,Normal
2126,RL,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,1Story,Gable,CompShg,VinylSd,VinylSd,Missing,TA,TA,PConc,Gd,TA,No,GLQ,Unf,GasA,Ex,Y,SBrkr,TA,Typ,Missing,Missing,Missing,Missing,Missing,Y,Missing,Missing,Missing,WD,Normal
1544,RM,Pave,Missing,Reg,Lvl,AllPub,Inside,Gtl,IDOTRR,Norm,Norm,1Fam,1Story,Gable,CompShg,VinylSd,VinylSd,Missing,TA,Gd,BrkTil,TA,TA,No,Rec,GLQ,GasA,TA,Y,SBrkr,TA,Typ,Missing,Detchd,Unf,TA,TA,P,Missing,MnPrv,Missing,WD,Normal
2415,RL,Pave,Missing,IR1,Lvl,AllPub,Inside,Gtl,Gilbert,Norm,Norm,1Fam,1Story,Gable,CompShg,VinylSd,VinylSd,Missing,Gd,TA,PConc,Gd,TA,Av,Unf,Unf,GasA,Ex,Y,SBrkr,Gd,Typ,Missing,Attchd,Fin,TA,TA,Y,Missing,Missing,Missing,New,Partial
2227,RM,Pave,Missing,IR1,Bnk,AllPub,Inside,Mod,Rare,Norm,Norm,TwnhsE,1Story,Gable,CompShg,HdBoard,HdBoard,Missing,TA,TA,CBlock,Gd,TA,Gd,GLQ,Unf,GasA,Gd,Y,SBrkr,TA,Typ,Missing,Attchd,Unf,TA,TA,Y,Missing,Missing,Missing,WD,Normal
410,RL,Pave,Missing,Reg,Lvl,AllPub,FR2,Gtl,Rare,Norm,Norm,Twnhs,2Story,Gable,CompShg,Plywood,Rare,Missing,TA,TA,CBlock,Gd,TA,No,ALQ,Unf,GasA,TA,Y,SBrkr,TA,Typ,TA,Attchd,Unf,TA,TA,Y,Missing,Missing,Missing,WD,Normal
761,RL,Pave,Missing,IR1,Lvl,AllPub,Inside,Gtl,Sawyer,Norm,Norm,1Fam,1Story,Gable,CompShg,MetalSd,MetalSd,Missing,TA,Gd,CBlock,Fa,TA,No,Unf,Unf,GasA,Ex,Y,FuseA,Fa,Typ,Missing,Rare,Unf,TA,TA,Y,Missing,Missing,Missing,WD,Normal
436,RL,Pave,Missing,Rare,Lvl,AllPub,CulDSac,Gtl,NridgHt,Norm,Norm,1Fam,2Story,Gable,CompShg,VinylSd,VinylSd,Stone,Gd,TA,PConc,Ex,TA,Av,GLQ,Unf,GasA,Ex,Y,SBrkr,Ex,Typ,Gd,BuiltIn,Fin,TA,TA,Y,Missing,Missing,Missing,WD,Family


In [ ]:
for feature in categorical_f:
    # 1. Average price per category, using train only.
    #    y_train has the prices; X_train[feature] tells which category each row is in.
    #    Sorted from cheapest category to most expensive.
    means = y_train.groupby(X_train[feature]).mean().sort_values()

    # 2. Give each category a number: cheapest = 0, next = 1, next = 2, ...
    labels_ordered = {k: i for i, k in enumerate(means.index, 0)}

    # 3. Replace the text with its number in BOTH train and test,
    #    using the same mapping learned from train.
    X_train[feature] = X_train[feature].map(labels_ordered)
    X_test[feature]  = X_test[feature].map(labels_ordered)

# 4. Categories that exist only in test have no number, so they become NaN.
#    Fill them with -1 (or the median) before scaling.
X_train[categorical_f] = X_train[categorical_f].fillna(-1)
X_test[categorical_f]  = X_test[categorical_f].fillna(-1)

In [ ]:
print(X_train.select_dtypes(exclude='number').columns)   # should be empty

Index([], dtype='object')


**Feature Scaling**

In [ ]:
from sklearn.preprocessing import MinMaxScaler


scaler = MinMaxScaler()

# fit on train only, then transform both
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),      # transform only, never fit on test
    columns=X_test.columns,
    index=X_test.index
)

In [ ]:
id_cols = ['Order', 'PID']   # change to your Id column names

# 1. Put the IDs back in front of the scaled features
train_final = pd.concat([df.loc[X_train_scaled.index, id_cols], X_train_scaled], axis=1)
test_final  = pd.concat([df.loc[X_test_scaled.index,  id_cols], X_test_scaled],  axis=1)

# 2. Add the target columns
train_final['SalePrice']     = y_train        # real price
test_final['SalePrice']      = y_test
train_final['SalePrice_log'] = y_train_log    # log price (what the model trains on)
test_final['SalePrice_log']  = y_test_log

train_final.head()

In [ ]:
train_final.to_csv('train_transformed.csv', index=False)
test_final.to_csv('test_transformed.csv', index=False)



```
one single file
```



In [ ]:
train_final['Set'] = 'train'
test_final['Set']  = 'test'

all_final = pd.concat([train_final, test_final])
all_final.to_csv('all_transformed.csv', index=False)